# EDA: nineteenFeaturesDf.csv

This notebook performs an initial chunked Exploratory Data Analysis (EDA) on `nineteenFeaturesDf.csv` and saves a summary JSON and plots into `analysis/eda_output`.

Run instructions:
- Install dependencies: `pip install pandas matplotlib seaborn` (if not already installed)
- Run all cells; the notebook is designed to be memory-efficient for large CSVs.


In [ ]:
# Section 1: Import Required Libraries
import os
import json
import math
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

print("Python version:", pd.__version__)
print("Seaborn version:", sns.__version__)

OUT_DIR = 'analysis/eda_output'
os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
# Section 2: Load & Inspect dataset (header only for quick inspect)
CSV_PATH = 'nineteenFeaturesDf.csv'
# Read only a small sample/first chunk to preview column names and dtypes
try:
    df_head = pd.read_csv(CSV_PATH, nrows=1000)
    print('Shape (sample 1k):', df_head.shape)
    display(df_head.head())
    print('\nColumns:', list(df_head.columns))
    display(df_head.dtypes)
except FileNotFoundError:
    print(f"CSV not found at {CSV_PATH}. Please place the file in the repo root.")


In [ ]:
# Helper: best-match column finder

def find_columns(columns):
    lower = [c.lower() for c in columns]
    def match(substrs):
        for s in substrs:
            for i,c in enumerate(lower):
                if s in c:
                    return columns[i]
        return None

    return {
        'event_time': match(['eventtime', 'event_time', 'time']),
        'event_name': match(['eventname', 'event_name', 'event']),
        'user_agent': match(['useragent', 'user_agent', 'agent']),
        'source_ip': match(['sourceip', 'source_ip', 'ipaddress', 'sourceipaddress']),
        'user_identity': match(['useridentity', 'user_identity', 'userarn', 'userarn', 'username']),
        'error_code': match(['errorcode', 'error_code', 'error']),
        'error_message': match(['errormessage', 'error_message', 'message']),
        'request_instance_type': match(['requestparametersinstancetype', 'instance', 'instancetype'])
    }


In [ ]:
# Section 3: Chunked summary function (stream-friendly)

def chunked_summary(csv_path, chunksize=100000):
    cols = list(pd.read_csv(csv_path, nrows=0).columns)
    colmap = find_columns(cols)

    total_rows = 0
    missing = Counter()
    top_counters = {c: Counter() for c in cols}
    numeric_stats = {}
    sample_rows = []
    event_times = []

    for i, chunk in enumerate(pd.read_csv(csv_path, chunksize=chunksize)):
        total_rows += len(chunk)
        # missing
        for c in cols:
            missing[c] += int(chunk[c].isna().sum() + chunk[c].astype(str).str.strip().eq('').sum())
        # sample
        if i == 0:
            sample_rows.extend(chunk.head(3).to_dict(orient='records'))
        sample_rows.extend(chunk.sample(n=min(2, len(chunk))).to_dict(orient='records'))

        # top values
        for c in cols:
            vals = chunk[c].dropna().astype(str)
            if not vals.empty:
                top_counters[c].update(vals.value_counts().to_dict())

        # numeric stats
        for c in chunk.select_dtypes(include=[np.number]).columns:
            s = pd.to_numeric(chunk[c], errors='coerce')
            if c not in numeric_stats:
                numeric_stats[c] = {
                    'count': 0,
                    'sum': 0.0,
                    'sumsq': 0.0,
                    'min': None,
                    'max': None,
                }
            stats = numeric_stats[c]
            v = s.dropna()
            stats['count'] += int(v.count())
            stats['sum'] += float(v.sum()) if not v.empty else 0.0
            stats['sumsq'] += float((v ** 2).sum()) if not v.empty else 0.0
            stats['min'] = float(v.min()) if stats['min'] is None else min(stats['min'], float(v.min()))
            stats['max'] = float(v.max()) if stats['max'] is None else max(stats['max'], float(v.max()))

        # event times
        et_col = colmap.get('event_time')
        if et_col and et_col in chunk.columns:
            parsed = pd.to_datetime(chunk[et_col], errors='coerce').dropna()
            if not parsed.empty:
                event_times.extend(parsed.dt.floor('D').astype('datetime64[ns]'))

    # assemble summary
    summary = {
        'total_rows': total_rows,
        'columns': cols,
        'missing': dict(missing),
        'top_values': {c: top_counters[c].most_common(20) for c in cols},
        'numeric_stats': {},
        'samples': sample_rows[:10],
        'column_matches': colmap,
    }

    for c, s in numeric_stats.items():
        if s['count'] > 0:
            mean = s['sum'] / s['count']
            var = (s['sumsq'] / s['count']) - (mean ** 2)
            std = math.sqrt(var) if var > 0 else 0.0
        else:
            mean = std = None
        summary['numeric_stats'][c] = {
            'count': s['count'],
            'min': s['min'],
            'max': s['max'],
            'mean': mean,
            'std': std,
        }

    # event time series
    if event_times:
        et_series = pd.Series(pd.to_datetime(event_times))
        summary['event_daily_counts'] = et_series.dt.date.value_counts().sort_index().to_dict()
    else:
        summary['event_daily_counts'] = {}

    return summary


In [ ]:
# Section 4: Run the chunked summary (adjust chunksize if needed)

csv_path = 'nineteenFeaturesDf.csv'
summary = None
try:
    summary = chunked_summary(csv_path, chunksize=100000)
    # Save JSON summary
    outp = os.path.join(OUT_DIR, 'eda_summary_notebook.json')
    with open(outp, 'w') as f:
        json.dump(summary, f, default=str, indent=2)
    print(f"Saved summary to {outp}")
except Exception as e:
    print('Error running chunked summary:', e)


In [ ]:
# Section 5: Quick summary checks
if summary is not None:
    print('Total rows:', summary.get('total_rows'))
    # Top columns by missing fraction
    total_rows = summary.get('total_rows')
    missing = summary.get('missing', {})
    miss_frac = {c: (missing.get(c, 0) / total_rows) for c in summary['columns']}
    top_missing = sorted(miss_frac.items(), key=lambda x: x[1], reverse=True)[:20]
    print('\nTop missing columns (top 20):')
    for k, v in top_missing:
        print(f"  {k}: {v:.2%}")

    # show sample rows
    print('\nSample rows:')
    display(pd.DataFrame(summary.get('samples')))

    # show top values for event_name and user_identity if present
    colmap = summary.get('column_matches', {})
    for key in ['event_name', 'user_identity', 'request_instance_type', 'error_code']:
        col = colmap.get(key)
        if col and summary['top_values'].get(col):
            print(f"\nTop values for {col}:")
            display(pd.DataFrame(summary['top_values'][col][:10], columns=['value', 'count']))
